### KNN Regression

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV  # Hyperparameter tuning

from sklearn.preprocessing import MinMaxScaler, StandardScaler  #Feature Scaling

from sklearn.neighbors import KNeighborsRegressor

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv('KNN_reg_outlet_sales.csv')
df

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.300,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.500,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.200,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.930,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,FDF22,6.865,Low Fat,0.056783,Snack Foods,214.5218,OUT013,1987,High,Tier 3,Supermarket Type1,2778.3834
8519,FDS36,8.380,Regular,0.046982,Baking Goods,108.1570,OUT045,2002,NaN,Tier 2,Supermarket Type1,549.2850
8520,NCJ29,10.600,Low Fat,0.035186,Health and Hygiene,85.1224,OUT035,2004,Small,Tier 2,Supermarket Type1,1193.1136
8521,FDN46,7.210,Regular,0.145221,Snack Foods,103.1332,OUT018,2009,Medium,Tier 3,Supermarket Type2,1845.5976


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


In [4]:
df.isna().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [5]:
len(df['Item_Identifier'].unique())

1559

In [6]:
df['Item_Identifier'].value_counts()

Item_Identifier
FDW13    10
FDG33    10
NCY18     9
FDD38     9
DRE49     9
         ..
FDY43     1
FDQ60     1
FDO33     1
DRF48     1
FDC23     1
Name: count, Length: 1559, dtype: int64

In [7]:
import warnings
warnings.filterwarnings("ignore")

#### 1. item weight

In [8]:
df['Item_Weight'].median()

12.6

In [9]:
df['Item_Weight'].fillna(df['Item_Weight'].median(), inplace=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                8523 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


#### 2. Item_Fat_Content

In [11]:
df['Item_Fat_Content'].value_counts()

Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

In [12]:
df['Item_Fat_Content'].replace({'Low Fat':0, 'Regular':1, 'LF':0, 'reg':1, 'low fat':0}, inplace=True)

In [13]:
df['Item_Fat_Content'].value_counts()

Item_Fat_Content
0    5517
1    3006
Name: count, dtype: int64

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                8523 non-null   float64
 2   Item_Fat_Content           8523 non-null   int64  
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(2), object(6)
memory usage: 799.2+ KB


#### Item_Type

In [15]:
len(df['Item_Type'].unique())

16

In [16]:
df['Item_Type'].value_counts()

Item_Type
Fruits and Vegetables    1232
Snack Foods              1200
Household                 910
Frozen Foods              856
Dairy                     682
Canned                    649
Baking Goods              648
Health and Hygiene        520
Soft Drinks               445
Meat                      425
Breads                    251
Hard Drinks               214
Others                    169
Starchy Foods             148
Breakfast                 110
Seafood                    64
Name: count, dtype: int64

In [17]:
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,0,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,1,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,0,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,1,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,0,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [18]:
Item_Type_df = pd.get_dummies(df['Item_Type'], prefix='Item_Type', drop_first= True)
Item_Type_df

,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,Item_Type_Dairy,Item_Type_Frozen Foods,Item_Type_Fruits and Vegetables,Item_Type_Hard Drinks,Item_Type_Health and Hygiene,Item_Type_Household,Item_Type_Meat,Item_Type_Others,Item_Type_Seafood,Item_Type_Snack Foods,Item_Type_Soft Drinks,Item_Type_Starchy Foods
0,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
3,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
8519,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
8520,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
8521,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False


#### Outlet_Identifier

In [19]:
df['Outlet_Identifier'].value_counts()

Outlet_Identifier
OUT027    935
OUT013    932
OUT049    930
OUT046    930
OUT035    930
OUT045    929
OUT018    928
OUT017    926
OUT010    555
OUT019    528
Name: count, dtype: int64

In [20]:
Outlet_Identifier_df = pd.get_dummies(df['Outlet_Identifier'], prefix='Outlet_Identifier')

In [21]:
Outlet_Identifier_df

,Outlet_Identifier_OUT010,Outlet_Identifier_OUT013,Outlet_Identifier_OUT017,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049
0,False,False,False,False,False,False,False,False,False,True
1,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,True
3,True,False,False,False,False,False,False,False,False,False
4,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8518,False,True,False,False,False,False,False,False,False,False
8519,False,False,False,False,False,False,False,True,False,False
8520,False,False,False,False,False,False,True,False,False,False
8521,False,False,False,True,False,False,False,False,False,False


#### Outlet_Size

In [22]:
df['Outlet_Size'].value_counts()

Outlet_Size
Medium    2793
Small     2388
High       932
Name: count, dtype: int64

In [23]:
df['Outlet_Size'].replace({'Small':0,"Medium":1, "High":2}, inplace=True)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                8523 non-null   float64
 2   Item_Fat_Content           8523 non-null   int64  
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   float64
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(5), int64(2), object(5)
memory usage: 799.2+ KB


In [25]:
df['Outlet_Size'].value_counts()

Outlet_Size
1.0    2793
0.0    2388
2.0     932
Name: count, dtype: int64

In [26]:
df['Outlet_Size'].mode()

0    1.0
Name: Outlet_Size, dtype: float64

In [27]:
df['Outlet_Size'].fillna(1, inplace=True)

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                8523 non-null   float64
 2   Item_Fat_Content           8523 non-null   int64  
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                8523 non-null   float64
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(5), int64(2), object(5)
memory usage: 799.2+ KB


#### Outlet_Location_Type

In [29]:
df['Outlet_Location_Type'].value_counts()

Outlet_Location_Type
Tier 3    3350
Tier 2    2785
Tier 1    2388
Name: count, dtype: int64

In [30]:
df['Outlet_Location_Type'].replace({'Tier 1':0, 'Tier 2':1, 'Tier 3':2}, inplace=True)

#### Outlet_Type

In [31]:
df['Outlet_Type'].value_counts()

Outlet_Type
Supermarket Type1    5577
Grocery Store        1083
Supermarket Type3     935
Supermarket Type2     928
Name: count, dtype: int64

In [32]:
Outlet_Type_df = pd.get_dummies(df['Outlet_Type'], prefix='Outlet_Type', 
                                drop_first=True)

In [33]:
Outlet_Type_df

,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,True,False,False
1,False,True,False
2,True,False,False
3,False,False,False
4,True,False,False
...,...,...,...
8518,True,False,False
8519,True,False,False
8520,True,False,False
8521,False,True,False


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                8523 non-null   float64
 2   Item_Fat_Content           8523 non-null   int64  
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                8523 non-null   float64
 9   Outlet_Location_Type       8523 non-null   int64  
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(5), int64(3), object(4)
memory usage: 799.2+ KB


In [35]:
df.drop(['Item_Identifier','Item_Type','Outlet_Identifier', 'Outlet_Type'], axis=1, inplace=True)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Weight                8523 non-null   float64
 1   Item_Fat_Content           8523 non-null   int64  
 2   Item_Visibility            8523 non-null   float64
 3   Item_MRP                   8523 non-null   float64
 4   Outlet_Establishment_Year  8523 non-null   int64  
 5   Outlet_Size                8523 non-null   float64
 6   Outlet_Location_Type       8523 non-null   int64  
 7   Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(5), int64(3)
memory usage: 532.8 KB


In [37]:
df_list = [df, Item_Type_df ,Outlet_Identifier_df, Outlet_Type_df]
df = pd.concat(df_list, axis=1)
df

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Outlet_Sales,Item_Type_Breads,Item_Type_Breakfast,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.300,0,0.016047,249.8092,1999,1.0,0,3735.1380,False,False,...,False,False,False,False,False,False,True,True,False,False
1,5.920,1,0.019278,48.2692,2009,1.0,2,443.4228,False,False,...,True,False,False,False,False,False,False,False,True,False
2,17.500,0,0.016760,141.6180,1999,1.0,0,2097.2700,False,False,...,False,False,False,False,False,False,True,True,False,False
3,19.200,1,0.000000,182.0950,1998,1.0,2,732.3800,False,False,...,False,False,False,False,False,False,False,False,False,False
4,8.930,0,0.000000,53.8614,1987,2.0,2,994.7052,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,0,0.056783,214.5218,1987,2.0,2,2778.3834,False,False,...,False,False,False,False,False,False,False,True,False,False
8519,8.380,1,0.046982,108.1570,2002,1.0,1,549.2850,False,False,...,False,False,False,False,True,False,False,True,False,False
8520,10.600,0,0.035186,85.1224,2004,0.0,1,1193.1136,False,False,...,False,False,False,True,False,False,False,True,False,False
8521,7.210,1,0.145221,103.1332,2009,1.0,2,1845.5976,False,False,...,True,False,False,False,False,False,False,False,True,False


In [38]:
df.head().T

,0,1,2,3,4
Item_Weight,9.3,5.92,17.5,19.2,8.93
Item_Fat_Content,0,1,0,1,0
Item_Visibility,0.016047,0.019278,0.01676,0.0,0.0
Item_MRP,249.8092,48.2692,141.618,182.095,53.8614
Outlet_Establishment_Year,1999,2009,1999,1998,1987
Outlet_Size,1.0,1.0,1.0,1.0,2.0
Outlet_Location_Type,0,2,0,2,2
Item_Outlet_Sales,3735.138,443.4228,2097.27,732.38,994.7052
Item_Type_Breads,False,False,False,False,False
Item_Type_Breakfast,False,False,False,False,False


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 36 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Item_Weight                      8523 non-null   float64
 1   Item_Fat_Content                 8523 non-null   int64  
 2   Item_Visibility                  8523 non-null   float64
 3   Item_MRP                         8523 non-null   float64
 4   Outlet_Establishment_Year        8523 non-null   int64  
 5   Outlet_Size                      8523 non-null   float64
 6   Outlet_Location_Type             8523 non-null   int64  
 7   Item_Outlet_Sales                8523 non-null   float64
 8   Item_Type_Breads                 8523 non-null   bool   
 9   Item_Type_Breakfast              8523 non-null   bool   
 10  Item_Type_Canned                 8523 non-null   bool   
 11  Item_Type_Dairy                  8523 non-null   bool   
 12  Item_Type_Frozen Foo

In [40]:
X = df.drop('Item_Outlet_Sales', axis=1)
y = df['Item_Outlet_Sales']

In [41]:
df

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Outlet_Sales,Item_Type_Breads,Item_Type_Breakfast,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.300,0,0.016047,249.8092,1999,1.0,0,3735.1380,False,False,...,False,False,False,False,False,False,True,True,False,False
1,5.920,1,0.019278,48.2692,2009,1.0,2,443.4228,False,False,...,True,False,False,False,False,False,False,False,True,False
2,17.500,0,0.016760,141.6180,1999,1.0,0,2097.2700,False,False,...,False,False,False,False,False,False,True,True,False,False
3,19.200,1,0.000000,182.0950,1998,1.0,2,732.3800,False,False,...,False,False,False,False,False,False,False,False,False,False
4,8.930,0,0.000000,53.8614,1987,2.0,2,994.7052,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,0,0.056783,214.5218,1987,2.0,2,2778.3834,False,False,...,False,False,False,False,False,False,False,True,False,False
8519,8.380,1,0.046982,108.1570,2002,1.0,1,549.2850,False,False,...,False,False,False,False,True,False,False,True,False,False
8520,10.600,0,0.035186,85.1224,2004,0.0,1,1193.1136,False,False,...,False,False,False,True,False,False,False,True,False,False
8521,7.210,1,0.145221,103.1332,2009,1.0,2,1845.5976,False,False,...,True,False,False,False,False,False,False,False,True,False


In [42]:
X

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.300,0,0.016047,249.8092,1999,1.0,0,False,False,False,...,False,False,False,False,False,False,True,True,False,False
1,5.920,1,0.019278,48.2692,2009,1.0,2,False,False,False,...,True,False,False,False,False,False,False,False,True,False
2,17.500,0,0.016760,141.6180,1999,1.0,0,False,False,False,...,False,False,False,False,False,False,True,True,False,False
3,19.200,1,0.000000,182.0950,1998,1.0,2,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,8.930,0,0.000000,53.8614,1987,2.0,2,False,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,0,0.056783,214.5218,1987,2.0,2,False,False,False,...,False,False,False,False,False,False,False,True,False,False
8519,8.380,1,0.046982,108.1570,2002,1.0,1,False,False,False,...,False,False,False,False,True,False,False,True,False,False
8520,10.600,0,0.035186,85.1224,2004,0.0,1,False,False,False,...,False,False,False,True,False,False,False,True,False,False
8521,7.210,1,0.145221,103.1332,2009,1.0,2,False,False,False,...,True,False,False,False,False,False,False,False,True,False


In [43]:
y

0       3735.1380
1        443.4228
2       2097.2700
3        732.3800
4        994.7052
          ...    
8518    2778.3834
8519     549.2850
8520    1193.1136
8521    1845.5976
8522     765.6700
Name: Item_Outlet_Sales, Length: 8523, dtype: float64

In [44]:
X.columns

Index(['Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Item_Type_Breads', 'Item_Type_Breakfast', 'Item_Type_Canned',
       'Item_Type_Dairy', 'Item_Type_Frozen Foods',
       'Item_Type_Fruits and Vegetables', 'Item_Type_Hard Drinks',
       'Item_Type_Health and Hygiene', 'Item_Type_Household', 'Item_Type_Meat',
       'Item_Type_Others', 'Item_Type_Seafood', 'Item_Type_Snack Foods',
       'Item_Type_Soft Drinks', 'Item_Type_Starchy Foods',
       'Outlet_Identifier_OUT010', 'Outlet_Identifier_OUT013',
       'Outlet_Identifier_OUT017', 'Outlet_Identifier_OUT018',
       'Outlet_Identifier_OUT019', 'Outlet_Identifier_OUT027',
       'Outlet_Identifier_OUT035', 'Outlet_Identifier_OUT045',
       'Outlet_Identifier_OUT046', 'Outlet_Identifier_OUT049',
       'Outlet_Type_Supermarket Type1', 'Outlet_Type_Supermarket Type2',
       'Outlet_Type_Supermarket Type3'],
      dtype='object')

In [45]:
df.describe()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Outlet_Sales
count,8523.00000,8523.000000,8523.000000,8523.000000,8523.000000,8523.000000,8523.000000,8523.000000
mean,12.81342,0.352693,0.066132,140.992782,1997.831867,0.829168,1.112871,2181.288914
std,4.22724,0.477836,0.051598,62.275067,8.371760,0.600327,0.812757,1706.499616
min,4.55500,0.000000,0.000000,31.290000,1985.000000,0.000000,0.000000,33.290000
25%,9.31000,0.000000,0.026989,93.826500,1987.000000,0.000000,0.000000,834.247400
50%,12.60000,0.000000,0.053931,143.012800,1999.000000,1.000000,1.000000,1794.331000
75%,16.00000,1.000000,0.094585,185.643700,2004.000000,1.000000,2.000000,3101.296400
max,21.35000,1.000000,0.328391,266.888400,2009.000000,2.000000,2.000000,13086.964800


In [46]:
norm_scaler = MinMaxScaler()
X_nor = norm_scaler.fit_transform(X)
X_nor
X_nor_df= pd.DataFrame(X_nor, columns=X.columns)
X_nor_df

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,0.282525,0.0,0.048866,0.927507,0.583333,0.5,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
1,0.081274,1.0,0.058705,0.072068,1.000000,0.5,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.770765,0.0,0.051037,0.468288,0.583333,0.5,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,0.871986,1.0,0.000000,0.640093,0.541667,0.5,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.260494,0.0,0.000000,0.095805,0.083333,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,0.137541,0.0,0.172914,0.777729,0.083333,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
8519,0.227746,1.0,0.143069,0.326263,0.708333,0.5,0.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
8520,0.359929,0.0,0.107148,0.228492,0.791667,0.0,0.5,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
8521,0.158083,1.0,0.442219,0.304939,1.000000,0.5,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### Train Test Split

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X_nor_df, y, test_size=0.15, random_state=42)

In [48]:
X_train.shape

(7244, 35)

In [49]:
X_test

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
7503,0.580232,0.0,0.080087,0.204332,0.083333,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2957,0.200953,0.0,0.216619,0.048466,0.500000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
7031,0.592141,1.0,0.125805,0.045651,0.583333,0.5,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
1084,0.479012,1.0,0.136322,0.604484,0.000000,0.5,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
856,0.335814,1.0,0.037930,0.705527,0.791667,0.0,0.5,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1349,0.422447,0.0,0.121747,0.832261,0.583333,0.5,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3018,0.479012,1.0,0.452928,0.525646,0.000000,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3992,0.681453,1.0,0.528221,0.536537,1.000000,0.5,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8465,0.681453,1.0,0.325735,0.633593,0.708333,0.5,0.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


### Train Model

In [50]:
knn_reg = KNeighborsRegressor()
knn_reg.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [51]:
y_pred = knn_reg.predict(X_test)
y_pred

array([1066.34528, 1159.02464, 1507.23804, ..., 2588.36408, 3177.59708,
        469.52216], shape=(1279,))

In [52]:
len(y_pred)

1279

In [53]:
y_test[:5]

7503    1743.0644
2957     356.8688
7031     377.5086
1084    5778.4782
856     2356.9320
Name: Item_Outlet_Sales, dtype: float64

In [54]:
y_pred[:5]

array([1066.34528, 1159.02464, 1507.23804, 4727.57948, 4319.97672])

### Evaluation

In [55]:
# Mean Squared error
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error is :", mse)

Mean Squared Error is : 1412029.559399245


In [56]:
# Root Mean Squared error
rmse = np.sqrt(mse)
print("Root Mean Squared Error is :", rmse)

Root Mean Squared Error is : 1188.288500070267


In [57]:
# Mean Absolute Error
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error is :", mae)

Mean Absolute Error is : 838.0733293823299


In [58]:
# Accuracy score
accuracy = r2_score(y_test, y_pred)
print("Accuarcy of knn model is :", accuracy)

Accuarcy of knn model is : 0.49811521380298474


### Hyperparameter Tuning

#### Grid SearchCV

In [59]:
k = np.arange(2,15)
p = [1,2]
hyp = {'n_neighbors': k, "p":p}

In [60]:
hyp

{'n_neighbors': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14]),
 'p': [1, 2]}

In [61]:
knn_gscv = KNeighborsRegressor()
best_knn_model = GridSearchCV(knn_gscv,hyp,cv = 5)
best_knn_model.fit(X_train, y_train)

,estimator,KNeighborsRegressor()
,param_grid,"{'n_neighbors': array([ 2, 3..., 12, 13, 14]), 'p': [1, 2]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,np.int64(7)


In [62]:
best_knn_model.best_params_

{'n_neighbors': np.int64(7), 'p': 2}

In [63]:
knn_reg1 = KNeighborsRegressor(n_neighbors=7, p=2)
knn_reg1.fit(X_train, y_train)

,n_neighbors,7
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [64]:
y_pred1 = knn_reg1.predict(X_test)
y_pred1

array([ 927.55451429, 1163.53305714, 1586.88674286, ..., 2289.40085714,
       2867.02991429,  421.35628571], shape=(1279,))

In [65]:
accuarcy  = r2_score(y_test, y_pred1)
accuarcy

0.5046524688790588

#### Randomized Search CV

In [66]:
k = np.arange(2,25)
p = [1,2]
hyp = {'n_neighbors': k, "p":p}

In [67]:
hyp

{'n_neighbors': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
        19, 20, 21, 22, 23, 24]),
 'p': [1, 2]}

In [68]:
knn_rscv = KNeighborsRegressor()
best_knn_model2 = RandomizedSearchCV(knn_rscv,hyp,cv=5)
best_knn_model2.fit(X_train, y_train)

,estimator,KNeighborsRegressor()
,param_distributions,"{'n_neighbors': array([ 2, 3..., 22, 23, 24]), 'p': [1, 2]}"
,n_iter,10
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [69]:
best_knn_model2.best_params_

{'p': 1, 'n_neighbors': np.int64(7)}

In [70]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                            test_size=0.2, random_state=42)

In [71]:
knn_reg2 = KNeighborsRegressor(n_neighbors=7, p=1)
knn_reg2.fit(X_train, y_train)

,n_neighbors,7
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,1
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [72]:
y_pred2 = knn_reg2.predict(X_test)
y_pred2

array([ 955.61322857,  934.30762857,  726.48291429, ...,  570.97105714,
        625.28131429, 1375.06722857], shape=(1705,))

In [73]:
# Accuracy score
accuracy = r2_score(y_test, y_pred2)
print("Accuarcy of knn model is :", accuracy)

Accuarcy of knn model is : 0.5501413462562807


In [74]:
import pickle
knn_model = KNeighborsRegressor(
    n_neighbors=7,
    p=1
)

knn_model.fit(X_train, y_train)
with open("knn_model.pkl", "wb") as file:
    pickle.dump(knn_model, file)


with open("scaler.pkl", "wb") as file:
    pickle.dump(norm_scaler, file)

feature_columns = X.columns.tolist()

with open("feature_columns.pkl", "wb") as file:
    pickle.dump(feature_columns, file)

preprocessing_data = {
    "item_weight_median": 12.6,
    "item_fat_mapping": {
        "Low Fat": 0,
        "Regular": 1,
        "LF": 0,
        "reg": 1,
        "low fat": 0
    },
    "feature_columns": feature_columns
}

with open("preprocessing.pkl", "wb") as file:
    pickle.dump(preprocessing_data, file)

y_pred = knn_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("======================================")
print("      KNN MODEL TRAINING COMPLETE")
print("======================================")

print(f"Mean Squared Error : {mse:.2f}")
print(f"Root Mean Squared Error : {rmse:.2f}")
print(f"Mean Absolute Error : {mae:.2f}")
print(f"R2 Score : {r2:.4f}")

print("\n======================================")
print("         PICKLE FILES CREATED")
print("======================================")

print("✅ knn_model.pkl")
print("✅ scaler.pkl")
print("✅ feature_columns.pkl")
print("✅ preprocessing.pkl")

print("\nAll pickle files created successfully!")

      KNN MODEL TRAINING COMPLETE
Mean Squared Error : 1222702.36
Root Mean Squared Error : 1105.76
Mean Absolute Error : 772.36
R2 Score : 0.5501

         PICKLE FILES CREATED
✅ knn_model.pkl
✅ scaler.pkl
✅ feature_columns.pkl
✅ preprocessing.pkl

All pickle files created successfully!
